# Phase 5: Final Submission

Load the best model, predict test set, generate `submission.csv`, validate format.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency,
    build_windows, compute_norm_stats, normalize, denormalize,
    write_submission, validate_submission,
)
from src.model import TrafficGNN
from src.train import build_adj_tensor, encode_texts

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj = load_adjacency()

mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
adj_t = build_adj_tensor(adj).to(DEVICE)

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = np.array(T1 + T2)

X_norm = normalize(X_all, mean, std)
Y_norm = normalize(Y_all, mean, std)

print(f"Train samples: {len(X_all)}")
print(f"Norm stats: mean={mean.mean():.1f} ± {std.mean():.1f}")

## Option A: Train fresh with best config

In [ ]:
from src import cache_save, cache_load

cached = cache_load("text_embeddings_all")
if cached is not None:
    T_emb_all = cached["all"]
    print(f"Loaded from cache: {T_emb_all.shape}")
else:
    T_emb_all = encode_texts(T_all.tolist())
    print(f"Encoded: {T_emb_all.shape}")
    cache_save("text_embeddings_all", all=T_emb_all)



## Option B: Load saved model from previous run

In [ ]:
# Uncomment to load a saved model instead of training fresh:
#
# import pickle
# run_dir = Path("../submissions/20260718_120000_gnn_d64_l4_add")
# with open(run_dir / "model_gnn.pkl", "rb") as f:
#     state_dict = pickle.load(f)
# model = TrafficGNN(
#     num_nodes=1260, in_steps=15, out_steps=3,
#     text_dim=384, d_model=64, num_layers=4, dropout=0.2,
#     use_text=True, fusion="add",
# ).to(DEVICE)
# model.load_state_dict(state_dict)
# model.eval()
# print("Model loaded")

## Predict & Submit

In [ ]:
test_hist, test_texts = load_test_data()
test_hist_norm = normalize(test_hist, mean, std)
test_emb = encode_texts(test_texts)

@torch.no_grad()
def predict(model, X, T_emb, adj_t, device, bs=64):
    model.eval()
    preds = []
    for i in range(0, len(X), bs):
        xb = torch.tensor(X[i:i+bs], dtype=torch.float32, device=device)
        tb = torch.tensor(T_emb[i:i+bs], dtype=torch.float32, device=device)
        preds.append(model(xb, tb, adj_t).cpu().numpy())
    yp_norm = np.concatenate(preds, axis=0)
    return denormalize(yp_norm, mean, std).clip(min=0)

test_preds = predict(model, test_hist_norm, test_emb, adj_t, DEVICE)
print(f"Predictions: {test_preds.shape}, range=[{test_preds.min():.1f}, {test_preds.max():.1f}]")

out_dir = write_submission(test_preds, label="final", models={"gnn": model.state_dict()})

In [ ]:
validate_submission(out_dir / "submission.csv")